### 💻 Day 10: The Structured "Contract" (Code)

Instead of asking the model to write "Action: search[query]", we define a **Schema**. This creates a "Contract"—the model *must* return a valid JSON object that fits your function.

In [6]:
!pip install groq
!pip install tavily
import os
from groq import Groq
from tavily import TavilyClient
from google.colab import userdata
import json

# 1. Setup
groq = Groq(api_key=userdata.get('GROQ_API_KEY'))
tavily = TavilyClient(api_key=userdata.get('TAVILY_API_KEY'))

# 2. Define the "Contract" (Tool Schema)
tools = [{
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Search the live web for real-time information.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query"}
            },
            "required": ["query"]
        }
    }
}]

def run_structured_agent(prompt):
    print(f"👤 User: {prompt}\n" + "="*40)

    # Step 1: Send prompt + tool definition to Model
    response = groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        tools=tools,
        tool_choice={"type": "function", "function": {"name": "web_search"}} # Explicitly choose the web_search tool
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    if tool_calls:
        for tool_call in tool_calls:
            # Step 2: Extract structured arguments
            function_args = json.loads(tool_call.function.arguments)
            query = function_args.get("query")

            # Step 3: Execute (The Code side of the contract)
            print(f"📡 [NATIVE CALL] Executing web_search with query: '{query}'")
            search_result = tavily.search(query=query, max_results=2)

            # Step 4: Final Synthesis
            # (In a full loop, you'd send this back to the model)
            print(f"✅ Observation: {search_result['results'][0]['content'][:200]}...")
    else:
        print(f"🤖 Response: {response_message.content}")

run_structured_agent("Who won the last general election in the UK?")

👤 User: Who won the last general election in the UK?
📡 [NATIVE CALL] Executing web_search with query: 'UK last general election results'
✅ Observation: | |  | First party | Second party | | --- | --- | --- | |  |  |  | | Leader | Boris Johnson | Jeremy Corbyn | | Party | Conservative "Conservative Party (UK)") | Labour "Labour Party (UK)") | | Leader...
